## Data cleaning and preparation

This notebook prepares the raw delivery data for analysis. 
The yearly files are combined, cleaned, and formatted to create 
a consistent dataset for further modelling.

## Libraries
 
Pandas and NumPy are used for data processing and numerical operations.

In [9]:
import pandas as pd
import numpy as np

## Listing all raw files

The raw delivery data is stored in separate yearly Excel files (2019 - 2025).

In [10]:
files = [
    "2019 lev presisjon.xlsx",
    "2020 lev presisjon.xlsx",
    "2021 lev presisjon.xlsx",
    "2022 lev presisjon.xlsx",
    "2023 lev presisjon.xlsx",
    "2024 lev presisjon.xlsx",
    "2025 lev presisjon.xlsx"
]

## Read each Excel file
Each dataset is loaded and preprocessed by standardising column names, and removing irrelevant columns.list.

In [11]:
allData = []
for file in files:
    data = pd.read_excel(file, sheet_name="Export")
    data.columns = data.columns.astype(str).str.strip()

    keepColumns = []
    for column in data.columns:
        if "Unnamed" not in column:
            keepColumns.append(column)
    data = data[keepColumns]

    allData.append(data)


## Combine all years into one dataset

After all yearly files have been read, they are combined into one dataset.This gives one complete dataset covering the full period used in the project. The Dato column is converted to datetime format because dates are needed later for filtering and feature engineering. The combined raw dataset is saved as dataset7years.csv, so the next steps can start from one common file.

In [ ]:
data = pd.concat(allData, ignore_index=True)
data["Dato"] = pd.to_datetime(data["Dato"], errors="coerce")

print("Dataset:", data.shape)
data.to_csv("dataset7years.csv", index=False, encoding="utf-8-sig")

## Load the combined dataset and prepare date columns

The combined dataset is loaded, and date columns are converted to datetime format for further analysis.

In [ ]:
data = pd.read_csv("dataset7years.csv")
data["Dato"] = pd.to_datetime(data["Dato"], errors="coerce")

if "Start Levering" in data.columns:
    data["Start Levering"] = pd.to_datetime(data["Start Levering"], errors="coerce")

if "Stopp Levering" in data.columns:
    data["Stopp Levering"] = pd.to_datetime(data["Stopp Levering"], errors="coerce")

## Convert important columns to numeric values

Important columns are converted to numeric format so they can be used in calculations and analysis. Values that cannot be converted are set as missing to avoid incorrect data.

In [ ]:
numberColumns = ["Kundenr", "Antall leveranser", "Planlagt leveringstid (min)", "Leveringstid (min)"]

for column in numberColumns:
    data[column] = pd.to_numeric(data[column], errors="coerce")

## Remove rows with missing essential values

Rows with missing values in essential columns are removed to ensure data quality. These records cannot be used in further analysis, feature engineering, or modelling.

In [ ]:
rowsBefore = len(data)

neededColumns = ["Dato", "Kundenr", "Rutenr", "Antall leveranser", "Planlagt leveringstid (min)", "Leveringstid (min)"]
data = data.dropna(subset=neededColumns).reset_index(drop=True)

print("Rows before:", rowsBefore)
print("Rows after:", len(data))

## Remove invalid customer IDs

The dataset should only include real customer deliveries that is complete and reliable. Therefore customer numbers below 10000 are treated as invalid or not relevant for this project, as it is internal numbers in the company. This filter removes records that can not represent normal customer deliveries, and helps reduce the noise in the dataset.

In [ ]:
rowsBefore = len(data)

validCustomers = data["Kundenr"] > 9999
data = data[validCustomers]

print("Customer filter befiore:", rowsBefore)
print("Customer filter after:", len(data))

## Keep realistic delivery counts

The number of deliveries should be within a realistic range. Rows with zero deliveries or extremely high delivery counts may represent registration errors or unusual cases. Only rows with 1 to 500 deliveries are kept. This keeps the dataset focused on normal operational delivery activity only.

In [ ]:
rowsBefore = len(data)
data = data[data["Antall leveranser"].between(1, 500)]

print("Delivery filter before:", rowsBefore)
print("Delivery filter after:", len(data))

## Keep normal weekdays

The project focuses on regular weekday delivery operations. Therefore only Monday to Friday are kept, because weekend deliveries follows different planning patterns and could make the model less consistent and. This makes the dataset more comparable across the period.

In [ ]:
weekDays = ["Mandag", "Tirsdag", "Onsdag", "Torsdag", "Fredag"]

data = data[data["Ukedag"].isin(weekDays)]

## Remove selected customer categories

Some customer categories are removed because they are not part of the normal delivery pattern in this project. Categories 7 and 18 are filtered out, to make the dataset more relevant for the forecasting and scenario analysis.

In [ ]:
rowsBefore = len(data)
badCats = [7, 18]
data = data[~data["Kundekategori_kd"].isin(badCats)]

print("Rows before category filter:", rowsBefore)
print("Rows after category filter:", len(data))

## Remove unrealistic delivery times

Very large or invalid delivery times will affect both the model and the later scenario analysis. Planned delivery time is therefore kept between 1 and 300 minutes, while actual delivery time is kept between 0 and 300 minutes. This removes extreme values while still keeping normal variation in the delivery process, and is a common error.

In [ ]:
rowsBefore = len(data)

maxTime = 300
data = data[data["Planlagt leveringstid (min)"].between(1, maxTime)]
data = data[data["Leveringstid (min)"].between(0, maxTime)]

print("Time filter before:", rowsBefore)
print("Time filter after:", len(data))

## Check start and stop delivery times

If both start and stop delivery times are available, they should follow a logical order. Rows where the start and stop time are equal, or where the stop time is before the start time, are removed. These rows are likely caused by registration errors, and keeping them would make the delivery time data less reliable.

In [ ]:
if "Start Levering" in data.columns and "Stopp Levering" in data.columns:
    rowsBefore = len(data)

    hasStart = data["Start Levering"].notna()
    hasStop = data["Stopp Levering"].notna()

    sameTime = data["Start Levering"] == data["Stopp Levering"]
    stopBeforeStart = data["Stopp Levering"] < data["Start Levering"]

    badRows = hasStart & hasStop & (sameTime | stopBeforeStart)
    data = data[~badRows]

    print("Start/stop filter before:", rowsBefore)
    print("Start/stop filter after:", len(data))

## Create delay variable

A new variable is created to compare actual delivery time with planned delivery time. This shows whether deliveries took longer or shorter than planned. The variable is useful for understanding the cleaned dataset before moving on to modelling and scenario testing.

In [ ]:
data["delayTime"] = (data["Leveringstid (min)"] - data["Planlagt leveringstid (min)"])
print(data["delayTime"].describe())

## Save the cleaned dataset

After all cleaning steps are complete, the final dataset is saved as merged_clean_7years.csv. This file is used in the next notebooks for exploratory analysis, feature engineering, model training and scenario testing. Saving the cleaned file also makes the workflow easier to repeat, because the later notebooks do not need to run the full raw data cleaning process again.

In [ ]:
data.to_csv("merged_clean_7years.csv", index=False, encoding="utf-8-sig")
print("Saved clean data:", data.shape)